# Chapter 9.3 - Language Models

A language model assigns probabilities to token sequences and predicts the next token from context. This notebook connects the probability chain rule to cross-entropy, perplexity, and two correct ways to partition one corpus into minibatches.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- factor a sequence probability into next-token conditional probabilities
- interpret cross-entropy and perplexity for language modeling
- construct random and sequential subsequence minibatches
- trace batch and time axes without guessing
- catch a target-shift bug that creates a meaningless copy task


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.3.0 The Problem This Notebook Solves

For tokens `x1, x2, ..., xT`, the probability chain rule gives

```text
P(x1, ..., xT) = P(x1) P(x2 | x1) ... P(xT | x1, ..., x[T-1])
```

A **language model** estimates these probabilities. During next-token training, each input position is paired with the token one step to its right. The statistical definition and the array alignment describe the same task.

An **n-gram model** shortens the context to the previous `n - 1` tokens. It is easy to count but cannot flexibly share knowledge across similar contexts. Neural language models learn distributed internal representations instead.


## 9.3.1 Learning a Count-Based Bigram Model

A bigram model estimates `P(next | current)`. Add-one smoothing gives every possible next token one artificial count, preventing a zero probability for unseen pairs.


In [ ]:
tokens = "time moves and time changes and time moves".split()
vocabulary = sorted(set(tokens))
token_to_id = {token: i for i, token in enumerate(vocabulary)}
V = len(vocabulary)
counts = torch.ones(V, V)

for current, following in zip(tokens[:-1], tokens[1:]):
    counts[token_to_id[current], token_to_id[following]] += 1

probabilities = counts / counts.sum(dim=1, keepdim=True)
time_row = probabilities[token_to_id["time"]]
print(dict(zip(vocabulary, time_row.tolist())))
assert torch.allclose(probabilities.sum(dim=1), torch.ones(V))
assert time_row[token_to_id["moves"]] > time_row[token_to_id["and"]]


## 9.3.2 Cross-Entropy and Perplexity

For the correct next token, negative log-likelihood is `-log(p_correct)`. Averaging it across tokens gives cross-entropy loss. **Perplexity** is

```text
perplexity = exp(average cross-entropy)
```

It can be read as an effective number of equally plausible choices. Perplexity 1 is perfect confidence on every correct token. A uniform guess over `V` tokens has perplexity `V`. Lower is better, but comparisons are meaningful only when tokenization and evaluation data match.


In [ ]:
def perplexity(correct_token_probabilities):
    probabilities = torch.as_tensor(correct_token_probabilities)
    if torch.any(probabilities <= 0) or torch.any(probabilities > 1):
        raise ValueError("probabilities must be in (0, 1]")
    cross_entropy = -torch.log(probabilities).mean()
    return torch.exp(cross_entropy)

perfect = perplexity([1.0, 1.0, 1.0])
uniform_four = perplexity([0.25, 0.25, 0.25])
mixed = perplexity([0.8, 0.4, 0.2])
print(perfect, uniform_four, mixed)
assert torch.allclose(perfect, torch.tensor(1.0))
assert torch.allclose(uniform_four, torch.tensor(4.0))
assert 1 < mixed < 4


## 9.3.3 Partitioning Sequences: The Universal Shift Contract

Suppose `X` contains token IDs with shape `(batch, time)`. Its label `Y` must have the same shape, and every position must satisfy

```text
Y[:, t] is the token immediately after X[:, t]
```

Random iteration samples subsequences from shuffled starting positions. Sequential iteration reshapes a contiguous stream into rows and moves forward in time. Random iteration weakens continuity between minibatches; sequential iteration makes it possible to carry recurrent state between neighboring minibatches.


In [ ]:
def random_sequence_batches(corpus, batch_size, num_steps):
    offset = random.randint(0, num_steps - 1)
    usable = corpus[offset:]
    starts = list(range(0, len(usable) - num_steps, num_steps))
    random.shuffle(starts)
    for i in range(0, len(starts), batch_size):
        batch_starts = starts[i : i + batch_size]
        if len(batch_starts) < batch_size:
            continue
        X = torch.stack([usable[j : j + num_steps] for j in batch_starts])
        Y = torch.stack([usable[j + 1 : j + num_steps + 1] for j in batch_starts])
        yield X, Y

corpus = torch.arange(30)
X_random, Y_random = next(random_sequence_batches(corpus, batch_size=2, num_steps=5))
print(X_random)
print(Y_random)
assert shape(X_random) == shape(Y_random) == (2, 5)
assert torch.equal(Y_random[:, :-1], X_random[:, 1:])


In [ ]:
def sequential_sequence_batches(corpus, batch_size, num_steps):
    offset = random.randint(0, num_steps)
    usable_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    Xs = corpus[offset : offset + usable_tokens].reshape(batch_size, -1)
    Ys = corpus[offset + 1 : offset + 1 + usable_tokens].reshape(batch_size, -1)
    for start in range(0, Xs.shape[1] - num_steps + 1, num_steps):
        yield Xs[:, start : start + num_steps], Ys[:, start : start + num_steps]

random.seed(0)
batches = list(sequential_sequence_batches(torch.arange(40), batch_size=2, num_steps=4))
X0, Y0 = batches[0]
X1, Y1 = batches[1]
print(X0, Y0, X1, sep="\n")
assert shape(X0) == shape(Y0) == (2, 4)
assert torch.equal(Y0, X0 + 1)
assert torch.equal(X1[:, 0], X0[:, -1] + 1)


## 9.3.4 Break It Deliberately: Labels Equal Inputs

If `Y = X`, the network learns to reproduce the visible current token rather than predict the next token. Accuracy can look excellent while the language-model objective is wrong. We require both a shape contract and a temporal shift contract.


In [ ]:
X = torch.tensor([[2, 4, 1, 3]])
wrong_Y = X.clone()
correct_Y = torch.tensor([[4, 1, 3, 0]])

assert shape(wrong_Y) == shape(correct_Y) == shape(X)
assert torch.equal(wrong_Y, X)
assert torch.equal(correct_Y[:, :-1], X[:, 1:])
try:
    assert torch.equal(wrong_Y[:, :-1], X[:, 1:])
except AssertionError:
    print("Caught: equal-shaped labels violate the next-token shift contract")


## 9.3.5 What This Notebook Does Not Do

It does not estimate serious perplexity from this tiny invented corpus. Reliable language-model evaluation needs a held-out corpus, consistent tokenization, sufficient data, and careful treatment of sequence boundaries. Here the goal is to make probability, loss, and minibatch alignment mechanically exact.


## 9.3 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. What does the probability chain rule contribute to language modeling?
2. Why is a zero-probability n-gram dangerous?
3. What perplexity does a uniform model over V tokens obtain?
4. What are the axes of a language-model minibatch in this notebook?
5. Why might sequential partitioning be useful for recurrent state?
